# Start

# 🚀 Phase 1 : Modèles de Filtrage Collaboratif (Collaborative Filtering)

## 🎯 Objectif du Notebook
L'objectif de cette étape est de construire et d'évaluer notre premier moteur de recommandation basé sur le **Filtrage Collaboratif**. 
Ce type d'algorithme ne regarde pas le contenu des articles, mais se base uniquement sur le comportement des lecteurs : *"Les utilisateurs qui ont lu cet article ont également lu ceux-ci"*.

Puisque nous travaillons avec des données implicites (Implicit Feedback = de simples clics, pas d'étoiles sur 5) et que notre application finale doit tourner sur une architecture Cloud Serverless (**Azure Functions**), l'enjeu majeur sera la **Scalabilité**.

## 🧠 Algorithmes Mis à l'Épreuve
Afin de justifier nos choix techniques d'ingénierie, nous allons confronter trois méthodes algorithmiques sur nos 3 millions d'interactions :

1. **KNN Users (Memory-Based) via `scikit-surprise`**
   * *Approche :* Calcul des plus proches voisins (similarité Cosinus).
   * *Risque identifié :* Explosion de la RAM due à la création de la gigantesque matrice Utilisateur-Utilisateur.
2. **SVD (Model-Based) via `scikit-surprise`**
   * *Approche :* Factorisation de matrice classique.
   * *Objectif :* Prouver que la factorisation résout le problème de RAM du KNN.
3. **ALS - Alternating Least Squares (Model-Based) via `implicit`**
   * *Approche :* Factorisation de matrices creuses (Sparse Matrices), ultra-optimisée en C++ et conçue spécifiquement pour les données implicites.
   * *Objectif :* Obtenir la meilleure Latence pour la mise en production.

## 📏 Métriques d'Évaluation (Critères d'acceptation)
Nous appliquerons un banc d'essai (Benchmark) divisé en deux grandes catégories :

### 1. Métriques de Performance (Métier)
| Métrique | Explication |
|----------|-------------|
| **Hit Ratio @ 5** | Vérifie si l'article réellement lu est présent dans le Top 5 recommandé (calculé en *Leave-One-Out* chronologique). |
| **MRR @ 5 (Mean Reciprocal Rank)** | Évalue la précision du classement en récompensant les modèles qui placent le bon article en haut de la liste. |
| **Couverture du Catalogue (Coverage)** | Mesure la diversité des recommandations en calculant le pourcentage d'articles uniques proposés par rapport au catalogue total. |

### 2. Métriques de Consommation (Cloud)
| Métrique | Explication |
|----------|-------------|
| **Latence d'inférence (Vitesse)** | Doit être inférieure à 200 ms par requête pour garantir l'instantanéité de l'API web. |
| **Empreinte Mémoire / RAM (Coût)** | La taille des matrices exportées doit tenir dans les petites instances Serverless Azure. |



### 3. Méthode d'évaluation (Validation Chronologique)
* **Le Leave-One-Out Chronologique :** Pour simuler la réalité (ne pas prédire le passé avec le futur), on utilise tout l'historique d'un utilisateur (sauf son tout dernier clic) pour entraîner le modèle. Le Test Set est constitué **uniquement** de ce dernier clic chronologique absolu.

* L'objectif de l'algorithme est donc d'analyser le passé pour deviner ce dernier article exact et le placer dans son Top 5 de recommandations.

In [1]:
# Installation des librairies indispensables
!pip install implicit scikit-surprise pyarrow

import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit
from surprise import Dataset, Reader, SVD, KNNBasic
import time
import os
from google.colab import drive

print("✅ Librairies installées avec succès !")
drive.mount('/content/drive')

# --- MODIFIE CE CHEMIN SELON TON GOOGLE DRIVE ---
# Par exemple : "/content/drive/MyDrive/P10_Data/clicks_full.parquet"
CLICKS_PATH = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated/clicks_full.parquet"

print("📂 Chargement du dataset complet...")
clicks_df = pd.read_parquet(CLICKS_PATH)
print(f"✅ Données chargées : {len(clicks_df)} interactions")


✅ Librairies installées avec succès !
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Chargement du dataset complet...
✅ Données chargées : 2988181 interactions


In [2]:
print("🛠️ Préparation du Train/Test Split (Dernier clic = Cible)...")

# On ne garde que les utilisateurs ayant au moins 2 clics
user_counts = clicks_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 2].index

df_valid = clicks_df[clicks_df['user_id'].isin(valid_users)].copy()
df_valid = df_valid.sort_values(['user_id', 'click_timestamp'])

# Le Test Set = le tout dernier clic chronologique de CHAQUE utilisateur
test_set = df_valid.groupby('user_id').tail(1)

# Le Train Set = tout le reste de l'historique
train_set = df_valid.drop(test_set.index)

print(f"✅ Matrice d'entraînement (Train) : {len(train_set)} clics")
print(f"✅ Cibles à deviner (Test) : {len(test_set)} utilisateurs")


🛠️ Préparation du Train/Test Split (Dernier clic = Cible)...
✅ Matrice d'entraînement (Train) : 2665284 clics
✅ Cibles à deviner (Test) : 322897 utilisateurs


# SVD

Le modèle **SVD (Singular Value Decomposition)** repose sur la factorisation de matrices. Le principe mathématique est de prendre l'énorme **matrice d'interactions (Utilisateurs x Articles)** et de la décomposer en deux matrices plus petites (les "facteurs latents") :
1. Une matrice contenant les profils des utilisateurs.
2. Une matrice contenant les caractéristiques des articles.

La prédiction de l'intérêt d'un utilisateur pour un article est alors simplement calculée par le produit scalaire (matriciel) de ces deux matrices.

Cependant, par conception technique, SVD a impérativement besoin de **notes explicites** (Ratings, ex: 1 à 5 étoiles) pour s'entraîner avec la librairie `scikit-surprise`. Puisque notre jeu de données ne contient que de l'Implicit Feedback (des clics), nous allons devoir fabriquer des "notes artificielles" pour forcer l'algorithme à accepter nos données.

In [3]:
from surprise import Dataset, Reader, SVD, KNNBasic

# -------------------------------------------------------------------
# FORMULE DE RATING : Basée sur le nombre de sessions uniques
# -------------------------------------------------------------------

# 1. On compte le nombre de 'session_id' uniques pour chaque paire (Utilisateur, Article)
interactions = train_set.groupby(['user_id', 'click_article_id'])['session_id'].nunique().reset_index(name='sessions_count')

# 2. On transforme ce nombre de sessions en note sur 5 étoiles (Plafond à 5)
# Ex: Lu dans 1 session = 1 étoile. Lu dans 3 sessions distinctes = 3 étoiles.
interactions['rating'] = interactions['sessions_count'].apply(lambda x: min(float(x), 5.0))

print("📊 Distribution des nouvelles notes (Basées sur les sessions) :")
print(interactions['rating'].value_counts())

# 3. Création du Dataset pour la librairie Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(interactions[['user_id', 'click_article_id', 'rating']], reader)
trainset_surprise = data.build_full_trainset()


📊 Distribution des nouvelles notes (Basées sur les sessions) :
rating
1.0    2597053
2.0      30015
3.0       1816
4.0        337
5.0        197
Name: count, dtype: int64


In [ ]:
import sys
import time

# --- Modèle 1 : SVD ---
print("🧠 Entraînement de SVD sur le train set...")
t0 = time.time()
model_svd = SVD(n_factors=50, random_state=42)
# SVD gère nativement les inners_IDs (raw_ID -> Inner_ID)
model_svd.fit(trainset_surprise)
print(f"✅ SVD entraîné en {time.time() - t0:.2f} secondes")

🧠 Entraînement de SVD sur le train set...
✅ SVD entraîné en 10.59 secondes


# KNN

# Fonctionnement de KNN et comment le fit s'effectue

   "Le modèle **KNN (K-Nearest Neighbors / Plus Proches Voisins)** est une approche dite *Memory-Based* (basée sur la mémoire). \n",
    "Contrairement à SVD ou ALS, il ne cherche pas à compresser ou \"comprendre\" les données via des facteurs latents. Son principe est beaucoup plus direct : il se base sur l'historique brut pour trouver des jumeaux de lecture (User-User) ou des articles souvent lus ensemble (Item-Item).\n",
    "\n",
    "**Comment s'effectue le `fit()` ?**\n",
    "Lorsqu'on appelle la fonction `.fit()` sur un modèle KNN, il n'y a pas d'entraînement au sens strict (pas de descente de gradient ou d'optimisation mathématique itérative). L'algorithme se contente de :\n",
    "1. Ingérer la totalité de la matrice d'interactions en RAM.\n",
    "2. Calculer d'avance une gigantesque **Matrice de Similarité** (par exemple avec la distance Cosinus). Si on a 300 000 utilisateurs, il tente de créer une matrice carrée de 300 000 x 300 000 pour noter à quel point chaque utilisateur ressemble à tous les autres.\n",
    "\n",
    "C'est cette étape de \"mémorisation\" qui pose un problème de **Scalabilité** majeur. Une matrice de similarité pour des centaines de milliers d'utilisateurs pèse théoriquement plusieurs centaines de Gigaoctets en RAM et fait inévitablement crasher les instances clouds standards. C'est ce que nous allons démontrer ci-dessous."
]

In [ ]:
# --- Modèle 2 : KNN Users ---
print("\n🧠 Entraînement de KNN...")
print("⚠️ ATTENTION : Crash mémoire quasi certain sur Colab (Matrice de Similarité trop énorme).")
try:
    sim_options = {'name': 'cosine', 'user_based': True}
    model_knn = KNNBasic(sim_options=sim_options, random_state=42)
    model_knn.fit(trainset_surprise)
    print("✅ KNN entraîné !")
except MemoryError as e:
    print(f"💥 DÉMONSTRATION RÉUSSIE : CRASH MÉMOIRE. Le KNN Memory-Based n'est pas scalable !")
except Exception as e:
    print(f"💥 ERREUR : {e}")


: 

: 

# ALS

Le modèle **ALS (Alternating Least Squares)** est une approche par factorisation de matrices (Model-Based), tout comme SVD (donc se base aussi sur un produit de deux matrices une pour les users et l'autre pour les articles).

Cependant, il est le standard de l'industrie pour les algorithmes confrontés à de l'**Implicit Feedback** (des clics, vues) sans notation explicite.

**Comment s'effectue le `fit()` ?**
L'algorithme ALS résout le problème de factorisation d'une manière radicalement différente de la descente de gradient classique (SGD) :
1. Il gèle d'abord la matrice des "Profils Utilisateurs" et calcule mathématiquement la matrice exacte des "Caractéristiques Articles" (par moindres carrés).
2. Ensuite, il gèle les "Articles" et résout mathématiquement les "Utilisateurs".
3. Il alterne ce processus (d'où *Alternating*) pendant N itérations jusqu'à converger vers l'erreur minimale.

**Les avantages de l'approche :**
* Le calcul est massivement parallélisable (multi-coeurs ou multi-serveurs via Spark), ce qui le rend ultra-rapide.
* L'algorithme remplace la note manquante par une notion de **"Confiance"** : plus un utilisateur clique souvent sur un article, plus le modèle a "confiance" dans l'association, ce qui correspond parfaitement à notre base de données GloboNews.

In [ ]:
import scipy.sparse as sparse
import implicit

# 1. ALS a besoin que les index aillent de 0 à N de manière continue
# C'est la fameuse notion de "Mapping" que l'on a documentée !
users = interactions['user_id'].astype("category")
items = interactions['click_article_id'].astype("category")

# ``.cat.codes`` : Pandas va automatiquement traduire chaque étiquette en un nombre entier continu qui commence par 0.
# Le tout premier user_id trouvé (ex: 98302) devient la ligne 0.
# Le suivant devient la ligne 1, etc, pour faire une suite de 0 à N, les nouveaux indices (0..N) s'appellent les `inners_IDs`.
user_codes = users.cat.codes
item_codes = items.cat.codes
clicks = interactions['sessions_count'].astype(float)

# 2. Création de la matrice Item-User puis User-Item
# sparse_item_user : Crée la matrice compressée initiale (Articles en lignes, Utilisateurs en colonnes).
# sparse_user_item : Fait une transposition mathématique (.T) pour avoir les Utilisateurs en lignes et les Articles en colonnes, 
# car c'est le format exact qu'exige la librairie implicit (le model ALS) pour s'entraîner.
# (C'est d'ailleurs pour cette raison qu'on sauvegarde les dictionnaires de traduction "mapping" en Pickle à la toute fin du notebook : 
# l'API de production en aura besoin pour retraduire la ligne 0 de la matrice en vrai user_id pour le client).
sparse_item_user = sparse.csr_matrix((clicks, (item_codes, user_codes)))
sparse_user_item = sparse_item_user.T.tocsr()


# 3. Entraînement de l'algorithme Alternating Least Squares
print("\n🧠 Entraînement d'ALS...")
t0 = time.time()
model_als = implicit.als.AlternatingLeastSquares(factors=50, iterations=15, regularization=0.1, random_state=42)
model_als.fit(sparse_user_item)
print(f"✅ ALS entraîné en {time.time() - t0:.2f} secondes")



🧠 Entraînement d'ALS...


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

✅ ALS entraîné en 24.86 secondes


# Évaluation

**Méthode d'évaluation du Benchmark**

Afin de désigner le vainqueur entre SVD et ALS pour une mise en production cloud, nous allons évaluer les modèles selon les critères définis précédemment. Voici comment la cellule suivante procède :

1. **Échantillonnage du Test Set :** Nous ne sélectionnons que 10 000 utilisateurs pour le test. Pourquoi ? Parce que SVD n'a pas de fonction native pour chercher les 5 meilleurs articles. 
Pour faire un Top 5, on doit forcer le SVD à prédire le score de l'utilisateur sur **l'intégralité du catalogue d'articles**, puis trier les résultats. C'est extrêmement lent, et le faire sur 300 000 utilisateurs prendrait des jours entiers.

2. **Calcul du Hit Ratio @ 5 :** Pour chaque utilisateur, on vérifie simplement si son dernier vrai clic (caché) est présent dans les 5 articles recommandés.

3. **Calcul du MRR @ 5 :** Si le vrai clic est trouvé, on regarde sa position. S'il est 1er, on marque 1 point. S'il est 2ème, 0.5 point, etc. Cela récompense la précision du tri.

4. **Calcul du Coverage (Couverture) :** On garde en mémoire chaque article recommandé. À la fin, on regarde quel pourcentage du catalogue a été exploré par le modèle (pour s'assurer qu'il ne recommande pas en boucle les 10 mêmes articles populaires).

5. **Latence et RAM :** On chronomètre le temps de prédiction moyen par utilisateur et on pèse les matrices de facteurs latents générées par chaque modèle."


**Pourquoi la RAM consommée seulement et pas le Peak RSS**


1. La RAM consommée (L'empreinte mémoire "au repos")
C'est ce qu'on a calculé dans le notebook avec `sys.getsizeof()`. Cela représente la taille "physique" stricte des matrices de résultats (Profils Utilisateurs et Articles) une fois l'entraînement terminé.
* **Pourquoi c'est important ?** C'est la métrique cruciale pour l'**Inférence** (ton API Cloud). Quand notre application Serverless se réveille, elle charge juste ces matrices en RAM pour faire des prédictions. Si l'empreinte fait 100 Mo, une petite instance Azure à 250 Mo de RAM suffira amplement.

2. Le Peak RSS (Resident Set Size maximal)
C'est le "pic absolu" de mémoire RAM que le système d'exploitation a dû allouer à ton script Python *à un instant T* pendant qu'il travaillait.
* Pendant l'entraînement (`.fit()`), l'algorithme a besoin de faire des brouillons : il copie la base de données, crée des matrices temporaires pour la descente de gradient, calcule des erreurs, etc. La mémoire gonfle brusquement, puis se vide quand le calcul est fini (Garbage Collector).
* **Pourquoi c'est important ?** C'est la métrique cruciale pour l'**Entraînement**. C'est exactement à cause de ce *Peak RSS* que le modèle KNN a crashé ! À la fin, la matrice de résultat de KNN n'aurait peut-être pas été si lourde, mais au moment de *calculer* les similarités croisées, le "brouillon" du calcul (Peak RSS) a demandé plusieurs centaines de Giga-octets et a fait exploser la machine.

**En résumé**
* Le **Peak RSS** conditionne la machine dont tu as besoin pour l'**entraînement** (ex: grosse machine Data Science).
* La **RAM consommée** conditionne le serveur dont tu as besoin pour la **production / l'API** (ex: petit Azure Function).

Pour notre MVP, il est pas nécéssaire de developper la pipeline MLOps pour re fit les matrices avec de nouvelles données (nouveaux articles / nouvels utilisateurs ajoutés) sinon il aurait été important d'étudier le peakRSS pour évaluer nos prerequis matériels pour l'architecture.

## Métriques de performances

In [17]:
# ==========================================
# 🏆 MATCH FINAL : SVD (Surprise) VS ALS (Implicit) 🏆
# ==========================================
import time
import sys
import numpy as np

print("📊 Lancement de l'évaluation sur 10000 utilisateurs du Test Set...\n")

# 1. On isole 10000 utilisateurs pour que le SVD pour gagner du temps
test_sample = test_set.sample(10000, random_state=42)
test_targets = dict(zip(test_sample['user_id'], test_sample['click_article_id']))
test_users = list(test_targets.keys())

# Variables de suivi
svd_hits = 0
als_hits = 0
svd_mrr_total = 0
als_mrr_total = 0
svd_latency_total = 0
als_latency_total = 0
svd_recommended_items = set()
als_recommended_items = set()

# On liste tous les id des articles connus de Surprise (RAW IDs)
all_items_raw = list(trainset_surprise._raw2inner_id_items.keys())

# Le Dictionnaire de Surprise : Pendant l'entraînement (.fit()), la librairie Surprise crée un dictionnaire "secret" 
# appelé "_raw2inner_id_items" (exemple: {284931: 0, 10453: 1, 99483: 2, ...}) Les clés (keys) sont les 
# Vrais IDs (Raw) et les valeurs sont les index internes (Inner) dont on a parlé tout à l'heure.

for idx, user in enumerate(test_users):
    target_item = test_targets[user]

    # ----------------------------------------
    # EVALUATION SVD (Surprise)
    # ----------------------------------------
    try:
        # On vérifie si l'user est connu. Si oui, pas de ValueError.
        _ = trainset_surprise.to_inner_uid(user)
        t0 = time.time()
        
        # SVD predict prend des IDs RAW (vrais IDs) car SVD n'a pas de fonction de recommandation TOP N contrairement à ALS
        predictions = [model_svd.predict(user, i).est for i in all_items_raw]
        
        # On trie pour garder les 5 meilleurs
        top_5_idx = np.argsort(predictions)[-5:][::-1]
        top_5_items_svd = [all_items_raw[i] for i in top_5_idx]
        svd_latency_total += (time.time() - t0)
    
        if target_item in top_5_items_svd:
            svd_hits += 1
            rank = top_5_items_svd.index(target_item) + 1
            svd_mrr_total += 1.0 / rank
        
        svd_recommended_items.update(top_5_items_svd)
    except ValueError:
        pass # Nouvel utilisateur inconnu pour SVD (Cold Start)
        
    # ----------------------------------------
    # EVALUATION ALS (Implicit)
    # ----------------------------------------
    try:
        user_code = users.cat.categories.get_loc(user)
        t0 = time.time()
        
        # ALS possède une fonction C++ native ultra rapide pour le Top N
        als_recs, _ = model_als.recommend(user_code, sparse_user_item[user_code], N=5)
        
        # On retraduit les index internes de l'ALS en vrais IDs d'articles
        top_5_items_als = items.cat.categories[als_recs].tolist()
        als_latency_total += (time.time() - t0)
        
        if target_item in top_5_items_als:
            als_hits += 1
            rank = top_5_items_als.index(target_item) + 1
            als_mrr_total += 1.0 / rank
            
        als_recommended_items.update(top_5_items_als)
    except KeyError:
        pass # Nouvel utilisateur (Cold Start)

# --- AFFICHAGE DES RÉSULTATS ---
print("="*50)
print("🏁 RÉSULTATS DU BENCHMARK (10000 utilisateurs)")
print("="*50)

print("\n🎯 Pertinence Métier (Hit Ratio @ 5) :")
print(f" - SVD : {(svd_hits / 10000) * 100:.2f} %")
print(f" - ALS : {(als_hits / 10000) * 100:.2f} %")

print("\n🎯 Précision du Classement (MRR @ 5) :")
print(f" - SVD : {(svd_mrr_total / 10000):.4f}")
print(f" - ALS : {(als_mrr_total / 10000):.4f}")

print("\n🌐 Diversité (Coverage) :")
total_items = len(items.cat.categories)
print(f" - SVD : {(len(svd_recommended_items) / total_items) * 100:.2f} %")
print(f" - ALS : {(len(als_recommended_items) / total_items) * 100:.2f} %")

print("\n⚡ Latence d'inférence (Temps de réponse / utilisateur) :")
print(f" - SVD : {(svd_latency_total / 10000) * 1000:.2f} ms")
print(f" - ALS : {(als_latency_total / 10000) * 1000:.2f} ms")

print("\n💾 Empreinte Mémoire (RAM estimée pour le déploiement) :")
# Taille des matrices générées par les modèles
size_svd = sys.getsizeof(model_svd.pu) + sys.getsizeof(model_svd.qi)
size_als = sys.getsizeof(model_als.user_factors) + sys.getsizeof(model_als.item_factors)
print(f" - SVD : {size_svd / (1024*1024):.2f} Mo")
print(f" - ALS : {size_als / (1024*1024):.2f} Mo")

📊 Lancement de l'évaluation sur 10000 utilisateurs du Test Set...

🏁 RÉSULTATS DU BENCHMARK (10000 utilisateurs)

🎯 Pertinence Métier (Hit Ratio @ 5) :
 - SVD : 0.01 %
 - ALS : 14.39 %

🎯 Précision du Classement (MRR @ 5) :
 - SVD : 0.0001
 - ALS : 0.0838

🌐 Diversité (Coverage) :
 - SVD : 0.40 %
 - ALS : 0.58 %

⚡ Latence d'inférence (Temps de réponse / utilisateur) :
 - SVD : 170.23 ms
 - ALS : 1.20 ms

💾 Empreinte Mémoire (RAM estimée pour le déploiement) :
 - SVD : 0.00 Mo
 - ALS : 69.41 Mo


## Métriques de consommation

## RAM

In [ ]:
print("\n💾 Empreinte Mémoire RÉELLE (RAM estimée pour le déploiement) :")

# 1. On utilise .nbytes pour SVD et ALS car ils ont réussi à s'entraîner
size_svd = model_svd.pu.nbytes + model_svd.qi.nbytes
size_als = model_als.user_factors.nbytes + model_als.item_factors.nbytes

# 2. Pour KNN, on calcule la taille de la matrice carrée (Utilisateurs x Utilisateurs x 8 octets)
n_users = trainset_surprise.n_users
size_knn_theorique = n_users * n_users * 8

print(f" - ALS : {size_als / (1024*1024):.2f} Mo (Gagnant !)")
print(f" - SVD : {size_svd / (1024*1024):.2f} Mo")
print(f" - KNN : {size_knn_theorique / (1024*1024*1024):.2f} Go  <-- (Taille théorique qui a causé le crash de la machine !)")



💾 Empreinte Mémoire RÉELLE (RAM estimée pour le déploiement) :
 - ALS : 69.41 Mo (Gagnant !)
 - SVD : 138.82 Mo
 - KNN : 776.82 Go  <-- (Taille théorique qui a causé le crash de la machine !)


## Latence

In [ ]:
# Tester une inférence avec chaque model et calculer la latence et le peak RSS

# Tester une inférence avec chaque model et calculer la latence et le peak RSS

In [15]:
import tracemalloc
import time
import numpy as np

print("🔍 TEST D'INFÉRENCE UNITAIRE (1 Utilisateur)")
test_user = test_users[0]
print(f"Utilisateur cible : {test_user}\n")

# --- Test SVD ---
print("--- Modèle SVD ---")
tracemalloc.start()
t0 = time.time()
try:
    _ = trainset_surprise.to_inner_uid(test_user)
    predictions = [model_svd.predict(test_user, i).est for i in all_items_raw]
    top_5_idx = np.argsort(predictions)[-5:][::-1]
    top_5_items_svd = [all_items_raw[i] for i in top_5_idx]
except ValueError:
    pass
svd_latency = (time.time() - t0) * 1000
current, peak_svd = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"⏱️ Latence : {svd_latency:.2f} ms")
print(f"📈 Peak Memory (RAM allouée pendant le calcul) : {peak_svd / 1024:.2f} Ko")


# --- Test ALS ---
print("\n--- Modèle ALS ---")
user_code = users.cat.categories.get_loc(test_user)

tracemalloc.start()
t0 = time.time()
als_recs, _ = model_als.recommend(user_code, sparse_user_item[user_code], N=5)
top_5_items_als = items.cat.categories[als_recs].tolist()
als_latency = (time.time() - t0) * 1000
current, peak_als = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"⏱️ Latence : {als_latency:.2f} ms")
print(f"📈 Peak Memory (RAM allouée pendant le calcul) : {peak_als / 1024:.2f} Ko")

🔍 TEST D'INFÉRENCE UNITAIRE (1 Utilisateur)
Utilisateur cible : 122708

--- Modèle SVD ---
⏱️ Latence : 570.76 ms
📈 Peak Memory (RAM allouée pendant le calcul) : 1546.84 Ko

--- Modèle ALS ---
⏱️ Latence : 2.94 ms
📈 Peak Memory (RAM allouée pendant le calcul) : 167.07 Ko


1. **La Latence (Temps de réponse) :**
   * SVD met environ **570 millisecondes** juste pour trouver le Top 5 d'un seul utilisateur. C'est lent, car on l'oblige à faire une prédiction manuelle sur tout le catalogue et à faire le tri lui-même.
   * ALS met seulement **3 millisecondes** !! Il est quasiment **200 fois plus rapide** car la librairie `implicit` est compilée en C++ ultra-optimisé avec des opérations mathématiques vectorisées. C'est instantané pour l'utilisateur de GloboNews.

2. **Le Peak Memory (Brouillon RAM pendant le calcul) :**
   * SVD alloue **1.5 Mo** de mémoire temporaire pour construire ses listes de prédiction pour tout le catalogue.
   * ALS n'utilise que **167 Ko** (soit presque 10 fois moins).

**Pourquoi cette différence ?**

Contrairement à ALS (`implicit`) qui est nativement optimisé en C++ pour générer un Top-N instantanément via une unique opération matricielle (vectorisation), la librairie SVD (`Surprise`) n'a pas de fonction Top-N intégrée. SVD est donc contraint informatiquement de parcourir **tout le catalogue d'articles** avec une boucle `for` en Python pour calculer puis trier manuellement les prédictions une par une, ce qui fait inévitablement exploser la latence et la RAM.

# Sauvegarde de la matrice ALS (Trainset) et ses mappings

In [16]:
import pickle
import numpy as np

print("💾 SAUVEGARDE DU MODÈLE GAGNANT (ALS) POUR LA PRODUCTION...")

# --- MODIFIE CE CHEMIN VERS LE DOSSIER DE TON DRIVE ---
SAVE_DIR = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated/ALS/"

# 1. Sauvegarde des matrices (Les "Cerveaux" du modèle) en format ultra-rapide .npy
np.save(f"{SAVE_DIR}als_user_factors.npy", model_als.user_factors)
np.save(f"{SAVE_DIR}als_item_factors.npy", model_als.item_factors)

# 2. Sauvegarde des Dictionnaires de traduction (Essentiel pour l'API)
# Pour retrouver à quel "vrai" user_id correspond la ligne 0, 1, 2...
with open(f"{SAVE_DIR}als_user_mapping.pkl", 'wb') as f:
    pickle.dump(users.cat.categories.values, f)
    
with open(f"{SAVE_DIR}als_item_mapping.pkl", 'wb') as f:
    pickle.dump(items.cat.categories.values, f)

print(f"✅ Modèle sauvegardé avec succès dans : {SAVE_DIR}")
print("Le notebook Filtrage Collaboratif est officiellement terminé !")

💾 SAUVEGARDE DU MODÈLE GAGNANT (ALS) POUR LA PRODUCTION...
✅ Modèle sauvegardé avec succès dans : /content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated/ALS/
Le notebook Filtrage Collaboratif est officiellement terminé !
